# Image Classification Using Vision Transformers (ViT's) Demonstration In Python

## **Background**

Vision Transformer (ViT) is a deep learning model that applies the Transformer architecture to image data. Instead of using convolutional layers like traditional CNNs, ViT splits an image into fixed-size patches, converts them into embeddings, and processes them using self-attention mechanisms. This enables the model to capture **global relationships across the entire image**, rather than focusing only on local spatial features.

ViT models are typically trained on large-scale datasets like ImageNet for **image classification tasks**, where the goal is to assign a **single label** to the entire image. Hence, the model learns to identify the **most dominant or visually salient object** in a scene. It does not inherently detect multiple objects or their locations unless extended for detection tasks.


## **Objective**

To demonstrate the use of a pre-trained Vision Transformer (ViT) model in Python for performing **image classification** on a real-world street image.



### Import Libraries

In [8]:
from transformers import ViTImageProcessor, ViTForImageClassification
from PIL import Image
import torch

import warnings
warnings.filterwarnings("ignore")

### Load Pretrained ViT Model

**`google/vit-base-patch16-224`**

* A pre-trained Vision Transformer (ViT) model from Google that splits images into **16×16 patches** and processes them using self-attention for image classification.
* Trained on **ImageNet (224×224 resolution)**, it predicts a **single dominant class** for an input image.


In [9]:
model_name = "google/vit-base-patch16-224"

model = ViTForImageClassification.from_pretrained(model_name)
processor = ViTImageProcessor.from_pretrained(model_name)

Loading weights: 100%|█████████████████████████████████████████████████████████████| 200/200 [00:00<00:00, 3099.19it/s]


### Load & Preprocess Image

In [10]:
image = Image.open("ViT Image.png")   # Use any image

inputs = processor(images=image, return_tensors="pt")

### Prediction

In [11]:
# Pass the input image through the ViT model
outputs = model(**inputs)

# Extract raw prediction scores (logits) for all 1000 classes
logits = outputs.logits

### Get Predicted Class


In [12]:
# Get index of the class with highest score (most probable class)
predicted_class_idx = logits.argmax(-1).item()

# Convert class index to actual label name
label = model.config.id2label[predicted_class_idx]

# Print final predicted class
print("Predicted Class:", label)

Predicted Class: traffic light, traffic signal, stoplight


### Interpretation

The model focuses on the most dominant object in the image (traffic light), and since it performs image-level classification, it does not identify smaller objects like the pedestrian.


###  Top predicted classes and their probabilities

Converts logits into probabilities using softmax, then selects the top 5 predicted classes.
Displays each class with its confidence score.


In [13]:
import torch.nn.functional as F

# Step 1: Convert logits to probabilities
probs = F.softmax(logits, dim=-1)

# Step 2: Get top-5 predictions
top5_probs, top5_indices = torch.topk(probs, k=5, dim=-1)

# Step 3: Display results
for i in range(top5_indices.shape[1]):
    idx = top5_indices[0][i].item()
    prob = top5_probs[0][i].item()

    label = model.config.id2label[idx]
    print(f"{i+1}. {label} : {round(prob * 100, 2)}%")

1. traffic light, traffic signal, stoplight : 96.52%
2. street sign : 2.8%
3. cab, hack, taxi, taxicab : 0.09%
4. pole : 0.04%
5. streetcar, tram, tramcar, trolley, trolley car : 0.02%


### Interpretation
The model strongly predicts **“traffic light” (96.52%)**, indicating high confidence that it is the dominant object in the image.

Other classes have very low probabilities, showing they are less relevant and the model is highly certain about its top prediction.


### Note :

Unlike a CNN, which must scan every detail piece-by-piece, a ViT looks at the whole image at once and uses Self-Attention to instantly "highlight" the most important part. It identifies the red light as the strongest global signal and mathematically "mutes" the person and car as background noise. This allows it to jump straight to a high-confidence Street Light label by focusing only on the most distinct clue in the entire frame.
